In [1]:
print("hello")

hello


In [ ]:
"""
Swiggy / Zomato — Data Cleaning Pipeline
=========================================
Import the raw CSV and apply:
  - Amount / order-count columns  → fill nulls with column MEAN
  - Other numeric columns         → fill nulls with column MEAN
  - Categorical columns           → fill nulls with column MODE
  - Integer-like columns          → cast back to whole numbers
"""

import pandas as pd
import numpy as np

# ──────────────────────────────────────────────────────────────
# STEP 1 — Import raw data
# ──────────────────────────────────────────────────────────────
RAW_PATH     = 'zomato_swiggy_raw.csv'    # ← change path if needed
CLEANED_PATH = 'zomato_swiggy_cleaned.csv'

df = pd.read_csv(RAW_PATH)

print("=" * 60)
print(f"Imported:  {RAW_PATH}")
print(f"Shape:     {df.shape[0]} rows × {df.shape[1]} columns")
print("=" * 60)
print("\nNull counts BEFORE cleaning:")
print(df.isnull().sum().to_string())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")


# ──────────────────────────────────────────────────────────────
# STEP 2 — Define column groups
# ──────────────────────────────────────────────────────────────

# Monetary + order-count → fill with MEAN
amount_order_cols = [
    'order_amount',        # total rupee value of the order
    'delivery_fee',        # delivery charge in rupees
    'discount_applied',    # coupon / discount in rupees
    'num_items',           # number of items ordered
    'num_orders_customer', # lifetime orders by this customer
]

# Other numeric → fill with MEAN
other_numeric_cols = [
    'rating',              # 1–5 star rating
    'delivery_time_min',   # delivery time in minutes
]

# Categorical / text → fill with MODE
categorical_cols = [
    'customer_id', 'platform', 'restaurant_name', 'cuisine_type',
    'city', 'payment_mode', 'order_status', 'is_first_order',
    'promo_code', 'customer_age_group',
]


# ──────────────────────────────────────────────────────────────
# STEP 3 — Amount & order-count columns  →  MEAN
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Amount & order-count columns  →  filled with MEAN")
print("=" * 60)

for col in amount_order_cols:
    if col not in df.columns:
        print(f"  [SKIP] '{col}' not found in dataset")
        continue
    mean_val = df[col].mean()
    null_cnt = df[col].isnull().sum()
    df[col]  = df[col].fillna(round(mean_val, 2))
    print(f"  {col:<25}  mean = {mean_val:>8.2f}   filled {null_cnt} nulls")


# ──────────────────────────────────────────────────────────────
# STEP 4 — Other numeric columns  →  MEAN
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Other numeric columns  →  filled with MEAN")
print("=" * 60)

for col in other_numeric_cols:
    if col not in df.columns:
        print(f"  [SKIP] '{col}' not found in dataset")
        continue
    mean_val = df[col].mean()
    null_cnt = df[col].isnull().sum()
    df[col]  = df[col].fillna(round(mean_val, 2))
    print(f"  {col:<25}  mean = {mean_val:>8.2f}   filled {null_cnt} nulls")


# ──────────────────────────────────────────────────────────────
# STEP 5 — Categorical columns  →  MODE
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Categorical columns  →  filled with MODE")
print("=" * 60)

for col in categorical_cols:
    if col not in df.columns:
        print(f"  [SKIP] '{col}' not found in dataset")
        continue
    mode_val = df[col].mode()[0]
    null_cnt = df[col].isnull().sum()
    df[col]  = df[col].fillna(mode_val)
    print(f"  {col:<25}  mode = '{mode_val}'   filled {null_cnt} nulls")


# ──────────────────────────────────────────────────────────────
# STEP 6 — Fix dtypes: integer-like columns → whole numbers
# ──────────────────────────────────────────────────────────────
int_cols = ['num_items', 'num_orders_customer', 'delivery_time_min']
for col in int_cols:
    if col in df.columns:
        df[col] = df[col].round(0).astype('Int64')


# ──────────────────────────────────────────────────────────────
# STEP 7 — Verify: no nulls remain
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Null counts AFTER cleaning:")
print("=" * 60)
print(df.isnull().sum().to_string())
print(f"\nTotal remaining nulls: {df.isnull().sum().sum()}")

print("\n" + "=" * 60)
print("Summary statistics — numeric columns (cleaned)")
print("=" * 60)
numeric_cols = amount_order_cols + other_numeric_cols
existing     = [c for c in numeric_cols if c in df.columns]
print(df[existing].describe().round(2).to_string())


# ──────────────────────────────────────────────────────────────
# STEP 8 — Save cleaned CSV
# ──────────────────────────────────────────────────────────────
df.to_csv(CLEANED_PATH, index=False)
print(f"\n✅  Cleaned file saved  →  {CLEANED_PATH}")
print(f"   Final shape: {df.shape[0]} rows × {df.shape[1]} columns")